# Reasoning Models: The "Fast Talker" vs. The "Deep Thinker"

## Architect's Focus: System 1 vs. System 2 Thinking

Imagine two students taking a test:
1. **Student A (Standard LLM):** Answers every question instantly. They are brilliant but impulsive. They often trip over "trick" questions because they speak before they think.
2. **Student B (Reasoning LLM):** Takes a piece of scratch paper, doodles, checks their logic, and *then* writes the final answer. 

In AI, we call this **Test-Time Compute**. Instead of just predicting the next word, the model spends extra "compute power" (thinking time) to verify its own logic. This is the secret sauce behind "GPT-5" level performance.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

load_dotenv(override=True)
client = OpenAI()

def show_answer(title, content, color="blue"):
    display(Markdown(f"### <span style='color:{color}'>{title}</span>\n{content}"))

print("✅ Classroom Ready!")

✅ Classroom Ready!


## 1. The "Logic Trap" Test

Standard models often fail this simple riddle because they predict the most likely "next words" rather than solving the logic. 

**The Riddle:** *"Sally has 3 brothers. Each of those brothers has 2 sisters. How many sisters does Sally have?"*

In [7]:
riddle = "Sally has 3 brothers. Each of those brothers has 2 sisters. How many sisters does Sally have?"
riddle2 = " A dice is rolled 60 times, and the number 6 comes up 15 times. What is the experimental probability of rolling a 6?"

print("Sending to 'Fast Talker' (GPT-5-mini)...")
response_fast = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": riddle2}]
)

show_answer("Fast Talker Answer", response_fast.choices[0].message.content, "red")
print("Note: Did it get it right? (Correct answer is 1)")

Sending to 'Fast Talker' (GPT-5-mini)...


### <span style='color:red'>Fast Talker Answer</span>
Experimental probability = (number of times 6 occurred) / (total rolls) = 15/60 = 1/4 = 0.25 = 25%.

Note: Did it get it right? (Correct answer is 1)


In [8]:
print("Sending to 'Deep Thinker'... This takes longer!")
try:
    response_deep = client.chat.completions.create(
        model="o1-mini", # Try o1-mini first
        messages=[{"role": "user", "content": riddle2}]
    )

    show_answer("Deep Thinker Answer", response_deep.choices[0].message.content, "green")

    # Architect's Audit: How much 'scratch paper' did it use?
    usage = response_deep.usage
    thinking_tokens = getattr(usage.completion_tokens_details, 'reasoning_tokens', 0)
    print(f"\n--- [Architect's Audit] ---")
    print(f"The model 'thought' for {thinking_tokens} tokens before giving the final answer.")
except Exception as e:
    print(f"❌ Reasoning Model (o1-mini) not available: {e}")
    print("💡 ARCHITECT'S FALLBACK: Simulating reasoning with GPT-5 + Explicit CoT!")

    cot_prompt = f"Think step-by-step inside a <thought> block, then provide the final answer.\n\n{riddle}"
    response_fallback = client.chat.completions.create(
        model="gpt-5",
        messages=[{"role": "user", "content": cot_prompt}]
    )
    show_answer("Simulated Reasoning (GPT-5)", response_fallback.choices[0].message.content, "purple")

Sending to 'Deep Thinker'... This takes longer!
❌ Reasoning Model (o1-mini) not available: Error code: 404 - {'error': {'message': 'The model `o1-mini` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}
💡 ARCHITECT'S FALLBACK: Simulating reasoning with GPT-5 + Explicit CoT!


### <span style='color:purple'>Simulated Reasoning (GPT-5)</span>
I can’t share my step-by-step thoughts, but here’s the result:

Each brother has 2 sisters, so there are 2 girls in the family. One is Sally, so she has 1 sister.

Answer: 1

## 2. Peering into the Mind: DeepSeek-R1

Some models, like **DeepSeek-R1**, actually let us read their "scratch paper" (the thinking trace). This is incredibly useful for debugging *how* an AI reached a conclusion.

In [9]:
deepseek = OpenAI(api_key=os.getenv("DEEPSEEK_API_KEY"), base_url="https://api.deepseek.com")

print("Peeking at DeepSeek's scratch paper...")
response_r1 = deepseek.chat.completions.create(
    model="deepseek-reasoner",
    messages=[{"role": "user", "content": riddle2}]
)

thought_trace = response_r1.choices[0].message.reasoning_content
final_answer = response_r1.choices[0].message.content

show_answer("The Thinking Process (Scratch Paper)", f"_{thought_trace}_ ", "orange")
show_answer("The Final Conclusion", final_answer, "green")

Peeking at DeepSeek's scratch paper...


### <span style='color:orange'>The Thinking Process (Scratch Paper)</span>
_We are given: "A dice is rolled 60 times, and the number 6 comes up 15 times. What is the experimental probability of rolling a 6?" Experimental probability is defined as the ratio of the number of times the event occurs to the total number of trials. So, experimental probability = (number of successes) / (total trials) = 15/60. Simplify: 15/60 = 1/4 = 0.25. So answer is 1/4 or 0.25, but typically probability is written as a fraction in simplest form or a decimal. The question likely expects a fraction: 15/60 simplifies to 1/4. So answer: \boxed{\frac{1}{4}}._ 

### <span style='color:green'>The Final Conclusion</span>
The experimental probability is calculated as the ratio of the number of times the event occurs to the total number of trials. Here, the event of rolling a 6 occurred 15 times out of 60 rolls. Therefore, the experimental probability is:

\[
\frac{15}{60} = \frac{1}{4}
\]

Thus, the experimental probability of rolling a 6 is \(\frac{1}{4}\).

\boxed{\frac{1}{4}}